# FIFA League Format Assistors Leaderboard

This notebook generates a best assistors leaderboard using FIFA league format with custom assists mapping and PAS (Passing) ratings from 60-99 scale.

## Section 1: Import Required Libraries

Import pandas, numpy for data processing and manipulation.

In [ ]:
import pandas as pd
import numpy as np
import os

# Set the working directory to player_stats folder
os.chdir(r'c:\Users\vardh\Desktop\MY.PROJECTS\prediction.oc\Prediction.oc\player_stats')

print("=== PROCESSING BEST ASSISTOR LEADERBOARD (EXACT FIFA LEAGUE FORMAT) ===")
print("Libraries imported successfully!")

## Section 2: Load Primary Statistics

Load player stats, shooting data, and standings information.

In [ ]:
# Load primary statistics
df_stats = pd.read_csv('cleaned_player_stats.csv')
df_shooting = pd.read_csv('cleaned_player_shooting.csv')
df_standings = pd.read_csv('raw/predicted_group_standings.csv')

print("All data files loaded successfully!")
print(f"Player stats shape: {df_stats.shape}")
print(f"Shooting data shape: {df_shooting.shape}")
print(f"Standings data shape: {df_standings.shape}")

## Section 3: Merge and Clean Data

Merge stats and shooting data, then clean null values for assists and xG assist.

In [ ]:
# Merge data across different datasets
df_merged = pd.merge(df_stats, df_shooting, on=['player', 'team'], suffixes=('_stats', '_shooting'), how='outer')
print(f"Merged data shape: {df_merged.shape}")

# Fill null values
df_merged['assists'] = df_merged['assists'].fillna(0)
df_merged['xg_assist'] = df_merged['xg_assist'].fillna(0)
df_merged['minutes'] = df_merged['minutes'].fillna(0)

print("\nData cleaning completed!")
print(f"Total assists in dataset: {df_merged['assists'].sum():.0f}")
print(f"Average xG Assist: {df_merged['xg_assist'].mean():.4f}")

## Section 4: Calculate Assistor Scores

Create composite assistor scores using balanced formula:
- Assists: 45% weight
- Expected Assisted Goals (xG Assist): 35% weight
- Minutes: 0.02% weight
- Team Points: 8% weight

In [ ]:
pts_map = df_standings.groupby('team')['points'].max().to_dict()

# Mathematical balanced formula for total creation index
df_merged['assistor_score'] = (
    (df_merged['assists'] * 0.45) + 
    (df_merged['xg_assist'] * 0.35) + 
    (df_merged['minutes'] * 0.0002) + 
    (df_merged['team'].map(lambda t: pts_map.get(t, 0)) * 0.08)
)

df_merged = df_merged.sort_values(by='assistor_score', ascending=False).reset_index(drop=True)
df_merged['rank'] = df_merged.index + 1

print("Assistor scores calculated!")
print(f"Score range: {df_merged['assistor_score'].min():.4f} to {df_merged['assistor_score'].max():.4f}")
print(f"Average score: {df_merged['assistor_score'].mean():.4f}")

## Section 5: Apply FIFA Assists Mapping

Apply precise assist mapping for top 20 players with desired values, and scale other players based on their assistor scores.

In [ ]:
# Apply precise alignment for the top 20 players as requested
desired_assists = {
    'Lionel Messi': 8, 'Antoine Griezmann': 7, 'Ivan Perišić': 6, 'Bruno Fernandes': 6,
    'Harry Kane': 5, 'Ousmane Dembélé': 5, 'Jordi Alba': 5, 'Kylian Mbappé': 5,
    'Theo Hernández': 4, 'Mislav Oršić': 4, 'Vinicius Júnior': 4, 'Leroy Sané': 4,
    'Alexis Mac Allister': 3, 'Enzo Fernández': 3, 'Lucas Paquetá': 3, 'Serge Gnabry': 3,
    'Raphinha': 3, 'Marcus Thuram': 3, 'Kaoru Mitoma': 2, 'César Azpilicueta': 2
}

print(f"Elite players mapped: {len(desired_assists)}")
print(f"Desired assists: {desired_assists}")

# Apply custom function to project the new assists
def map_assists(row):
    if row['player'] in desired_assists:
        return desired_assists[row['player']]
    min_score = df_merged['assistor_score'].min()
    max_score = df_merged['assistor_score'].max()
    return int(np.round(((row['assistor_score'] - min_score) / (max_score - min_score)) * 2))

df_merged['fifa_assists'] = df_merged.apply(map_assists, axis=1)

print(f"\nFIFA assists mapping completed!")
print(f"Assists range: {df_merged['fifa_assists'].min()} to {df_merged['fifa_assists'].max()}")

## Section 6: Generate FIFA PAS Ratings

Convert the overall index to a 60-99 FIFA Passing (PAS) card rating scale.

In [ ]:
# Convert the overall index to a 60-99 FIFA Passing (PAS) card rating
min_score = df_merged['assistor_score'].min()
max_score = df_merged['assistor_score'].max()
df_merged['pas_rating'] = 60 + ((df_merged['assistor_score'] - min_score) / (max_score - min_score)) * (99 - 60)
df_merged['pas_rating'] = np.round(df_merged['pas_rating']).astype(int)

print("FIFA PAS ratings generated!")
print(f"PAS rating range: {df_merged['pas_rating'].min()} to {df_merged['pas_rating'].max()}")
print(f"Average PAS rating: {df_merged['pas_rating'].mean():.1f}")

## Section 7: Format and Save Final Leaderboard

Create the final FIFA league format leaderboard and export to CSV with top 20 display.

In [ ]:
position_col = 'position_stats' if 'position_stats' in df_merged.columns else 'position'
df_merged = df_merged.rename(columns={position_col: 'position', 'minutes': 'minutes_played'})

# Replace the old assists column with the new desired values
if 'assists' in df_merged.columns:
    df_merged = df_merged.drop(columns=['assists'])
df_merged = df_merged.rename(columns={'fifa_assists': 'assists'})

final_columns = ['rank', 'player', 'team', 'position', 'assists', 'xg_assist', 'minutes_played', 'pas_rating']
leaderboard = df_merged[[col for col in final_columns if col in df_merged.columns]]

# Store as CSV
output_file = 'best_assistors_leaderboard.csv'
leaderboard.to_csv(output_file, index=False)

print(f"✓ Leaderboard saved to: {output_file}")

# Display top 20
print("\n" + "="*120)
print("TOP 20 BEST ASSISTORS (FIFA LEAGUE FORMAT)")
print("="*120)
display_cols = ['rank', 'player', 'team', 'position', 'assists', 'xg_assist', 'pas_rating']
top_20 = leaderboard[display_cols].head(20)
print(top_20.to_string(index=False))

# Additional statistics
print("\n" + "="*120)
print("LEADERBOARD STATISTICS")
print("="*120)
print(f"Total players ranked: {len(leaderboard)}")
print(f"Average assists: {leaderboard['assists'].mean():.2f}")
print(f"Average PAS rating: {leaderboard['pas_rating'].mean():.1f}")
print(f"PAS rating range: {leaderboard['pas_rating'].min()} to {leaderboard['pas_rating'].max()}")

print("\nProcessing Complete! ✓")